# Cross-Modal Spectral Prediction: AION-1 Benchmark Results

Evaluates AION-1's ability to predict DESI galaxy spectra from Legacy Survey images alone,
using the ProVABGS + DESI + Legacy Survey cross-matched dataset.

In [ ]:
import json
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})

REPO_ROOT = os.path.dirname(os.getcwd())  # assumes notebook is run from notebooks/
GT_PATH = os.path.join(REPO_ROOT, "artifacts", "ground_truth.hdf5")
PRED_PATH = os.path.join(REPO_ROOT, "artifacts", "predictions.hdf5")
METRICS_PATH = os.path.join(REPO_ROOT, "artifacts", "metrics.json")
PER_OBJ_PATH = os.path.join(REPO_ROOT, "artifacts", "metrics_per_object.parquet")
PER_WAV_PATH = os.path.join(REPO_ROOT, "artifacts", "per_wavelength.npz")

In [ ]:
# Load all data
with h5py.File(GT_PATH, "r") as f:
    images = f["images"][:]
    true_flux = f["spectrum_flux"][:]
    ivar = f["spectrum_ivar"][:]
    spec_mask = f["spectrum_mask"][:].astype(bool)
    wavelength = f["spectrum_lambda"][:]
    z_spec = f["z_spec"][:]

if wavelength.ndim == 2:
    wavelength = wavelength[0]

with h5py.File(PRED_PATH, "r") as f:
    pred_image_only = f["pred_flux_image_only"][:]
    pred_image_phot = f["pred_flux_image_phot"][:]

with open(METRICS_PATH) as f:
    metrics = json.load(f)

per_obj = pd.read_parquet(PER_OBJ_PATH)
per_wav = np.load(PER_WAV_PATH)

# Redshift bins (same logic as evaluate.py)
z_bins = per_obj["z_bin"].values

print(f"Loaded {len(true_flux)} galaxies, {len(wavelength)} wavelength bins")
print(f"Redshift bins: {dict(zip(*np.unique(z_bins, return_counts=True)))}")

## Plot 1: Example Gallery

One galaxy per redshift bin: Legacy Survey image + predicted vs ground-truth spectrum.

In [ ]:
def make_rgb(image_griz):
    """Make an RGB composite from g,r,i,z bands (use z,r,g -> R,G,B)."""
    rgb = np.stack([image_griz[3], image_griz[1], image_griz[0]], axis=-1)
    for c in range(3):
        lo, hi = np.percentile(rgb[..., c], [1, 99.5])
        rgb[..., c] = np.clip((rgb[..., c] - lo) / (hi - lo + 1e-8), 0, 1)
    return rgb ** 0.5  # gamma stretch


EMISSION_LINES = {
    "OII": 3727, "Hb": 4861, "OIII": 5007, "Ha": 6563, "NII": 6583, "SII": 6724,
}

# Pick one representative galaxy per redshift bin (median chi2 within each bin)
Z_BINS = sorted(set(z_bins))
n_bins = len(Z_BINS)

fig, axes = plt.subplots(n_bins, 3, figsize=(16, 3.5 * n_bins), gridspec_kw={"width_ratios": [1, 3, 3]})
if n_bins == 1:
    axes = axes[np.newaxis, :]

chi2_img_vals = per_obj["chi2_image_only"].values
chi2_phot_vals = per_obj["chi2_image_phot"].values

for row_i, zbin in enumerate(Z_BINS):
    mask = z_bins == zbin
    if not mask.any():
        continue

    type_indices = np.where(mask)[0]
    median_chi2 = np.nanmedian(chi2_img_vals[type_indices])
    idx = type_indices[np.nanargmin(np.abs(chi2_img_vals[type_indices] - median_chi2))]
    z = z_spec[idx]

    # Column 0: Image
    ax_img = axes[row_i, 0]
    rgb = make_rgb(images[idx])
    ax_img.imshow(rgb, origin="lower")
    ax_img.set_title(f"{zbin}  z={z:.3f}")
    ax_img.axis("off")

    # Column 1: Predicted (image-only) vs true spectrum
    ax_spec = axes[row_i, 1]
    valid = ~spec_mask[idx]
    ax_spec.plot(wavelength[valid], true_flux[idx, valid], color="gray", lw=0.5, alpha=0.7, label="Ground truth")
    ax_spec.plot(wavelength[valid], pred_image_only[idx, valid], color="steelblue", lw=0.7, alpha=0.9, label="Predicted (img)")
    ax_spec.set_xlim(3600, 9800)
    ax_spec.set_ylabel("Flux")
    ax_spec.set_title(f"Image only  chi2={chi2_img_vals[idx]:.2f}")
    if row_i == 0:
        ax_spec.legend(fontsize=8)
    for name, rest in EMISSION_LINES.items():
        obs = rest * (1 + z)
        if 3600 < obs < 9800:
            ax_spec.axvline(obs, color="red", alpha=0.2, lw=0.5)

    # Column 2: Predicted (image+phot) vs true
    ax_phot = axes[row_i, 2]
    ax_phot.plot(wavelength[valid], true_flux[idx, valid], color="gray", lw=0.5, alpha=0.7, label="Ground truth")
    ax_phot.plot(wavelength[valid], pred_image_phot[idx, valid], color="darkorange", lw=0.7, alpha=0.9, label="Predicted (img+phot)")
    ax_phot.set_xlim(3600, 9800)
    ax_phot.set_title(f"Image+phot  chi2={chi2_phot_vals[idx]:.2f}")
    if row_i == 0:
        ax_phot.legend(fontsize=8)
    for name, rest in EMISSION_LINES.items():
        obs = rest * (1 + z)
        if 3600 < obs < 9800:
            ax_phot.axvline(obs, color="red", alpha=0.2, lw=0.5)

    if row_i == n_bins - 1:
        axes[row_i, 1].set_xlabel("Wavelength [A]")
        axes[row_i, 2].set_xlabel("Wavelength [A]")

plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, "artifacts", "gallery.png"), dpi=150, bbox_inches="tight")
plt.show()

## Plot 2: Per-Wavelength Chi-Squared

Shows where in wavelength space the predictions are best/worst.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

wav = per_wav["wavelength"]

# Chi2 per wavelength
ax1.plot(wav, per_wav["chi2_image_only"], color="steelblue", lw=0.7, label="Image only")
ax1.plot(wav, per_wav["chi2_image_phot"], color="darkorange", lw=0.7, label="Image+phot")
ax1.axhline(1.0, color="black", ls="--", lw=0.5, alpha=0.5)
ax1.set_ylabel("Mean reduced chi2")
ax1.set_yscale("log")
ax1.legend()
ax1.set_title("Per-Wavelength Reduced Chi-Squared")

# Camera boundaries (approximate)
for boundary in [4500, 5900, 7500]:
    ax1.axvline(boundary, color="gray", ls=":", lw=0.5, alpha=0.5)

# Mean normalized residual (bias)
ax2.plot(wav, per_wav["mean_norm_residual_image_only"], color="steelblue", lw=0.5, alpha=0.8)
ax2.plot(wav, per_wav["mean_norm_residual_image_phot"], color="darkorange", lw=0.5, alpha=0.8)
ax2.axhline(0, color="black", ls="--", lw=0.5)
ax2.set_ylabel("Mean norm. residual")
ax2.set_xlabel("Wavelength [A]")
ax2.set_title("Systematic Bias (should be ~0 everywhere)")
ax2.set_xlim(3600, 9800)

plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, "artifacts", "chi2_per_wavelength.png"), dpi=150, bbox_inches="tight")
plt.show()

## Plot 3: Stratified Summary Table

Metrics by redshift bin and inference mode.

In [ ]:
# Build a combined summary table for both modes
rows = []
for mode_key, mode_label in [("image_only", "Image Only"), ("image_phot", "Image + Phot")]:
    for entry in metrics[mode_key]["summary"]:
        rows.append({
            "Mode": mode_label,
            "Stratum": entry["stratum"],
            "N": entry["n_objects"],
            "Median chi2": f"{entry['median_chi2']:.2f}",
            "Frac good": f"{entry['frac_good']:.1%}",
            "Median cont. R2": f"{entry['median_continuum_r2']:.3f}",
        })

summary_df = pd.DataFrame(rows)
summary_df.style.set_caption("Tier 1 Metrics by Redshift Bin and Inference Mode")

## Plot 4: Residual Heatmap

Objects sorted by redshift (y) vs wavelength (x), colored by normalized residual.
Reveals redshift-dependent systematic patterns.

In [ ]:
# Sort by redshift
sort_idx = np.argsort(z_spec)

valid = ~spec_mask & (ivar > 0)
norm_resid = (pred_image_only - true_flux) * np.sqrt(np.where(valid, ivar, 0))
norm_resid[~valid] = np.nan

# Subsample for visualization (max 500 rows)
step = max(1, len(sort_idx) // 500)
display_idx = sort_idx[::step]

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(
    norm_resid[display_idx],
    aspect="auto",
    cmap="RdBu_r",
    vmin=-3, vmax=3,
    extent=[wavelength[0], wavelength[-1], z_spec[display_idx[-1]], z_spec[display_idx[0]]],
    interpolation="nearest",
)
ax.set_xlabel("Wavelength [A]")
ax.set_ylabel("Redshift")
ax.set_title("Normalized Residual (Image Only): (pred - true) * sqrt(ivar)")
plt.colorbar(im, ax=ax, label="Normalized residual", shrink=0.8)

# Overlay emission line tracks
for name, rest in EMISSION_LINES.items():
    z_range = np.linspace(z_spec[display_idx[0]], z_spec[display_idx[-1]], 100)
    obs_wav = rest * (1 + z_range)
    in_range = (obs_wav > 3600) & (obs_wav < 9800)
    ax.plot(obs_wav[in_range], z_range[in_range], "k--", lw=0.5, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, "artifacts", "residual_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

## Plot 5: Chi-Squared Distribution by Redshift Bin

In [ ]:
z_bin_list = sorted(set(z_bins))
n_bins = len(z_bin_list)

fig, axes = plt.subplots(1, n_bins, figsize=(4 * n_bins, 3.5), sharey=True)
if n_bins == 1:
    axes = [axes]

for ax, zbin in zip(axes, z_bin_list):
    mask = per_obj["z_bin"] == zbin
    chi2_img = per_obj.loc[mask, "chi2_image_only"].dropna()
    chi2_phot = per_obj.loc[mask, "chi2_image_phot"].dropna()

    bins = np.logspace(-1, 2.5, 50)
    ax.hist(chi2_img, bins=bins, alpha=0.5, color="steelblue", label="Image only")
    ax.hist(chi2_phot, bins=bins, alpha=0.5, color="darkorange", label="Image+phot")
    ax.axvline(3.0, color="black", ls="--", lw=0.8, alpha=0.6)
    ax.set_xscale("log")
    ax.set_xlabel("Reduced chi2")
    ax.set_title(f"{zbin} (n={mask.sum()})")
    if ax == axes[0]:
        ax.set_ylabel("Count")
        ax.legend(fontsize=7)

plt.suptitle("Per-Object Reduced Chi-Squared Distribution", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, "artifacts", "chi2_histogram.png"), dpi=150, bbox_inches="tight")
plt.show()